# Strategy research

A playground for testing strategy ideas on real Kraken data without touching the live system. Everything here uses the
same strategy code the runtime runs, so a result you find here is a result of the strategy you would deploy.

- **Method and how to read results:** `docs/research_guide.md` (read the "Reading the output" section once).
- **Latest findings:** `docs/research_log.md`.
- **Same thing from the terminal:** `python scripts/research/research.py --help`.

Kernel: the `CryptoArb` conda env. The first run downloads data from Kraken's public API and caches it in
`data/historical_cache/`; after that it works offline.

## 0. Setup

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # data/ paths resolve the same way as for the CLI

import pandas as pd

from src.backtest.indicators import atr, rolling_max, rolling_mean, rolling_min, rsi, series, shift
from src.backtest.strategies import latch_position
from src.research import CATALOG, CostSettings, catalog_table, compare, load_bars, plot_heatmap, plot_run, run_strategy, summarize, sweep

%matplotlib inline
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

## 1. Data

Kraken's public API only returns the most recent 720 candles, so the interval decides how far back you can see:
`"4h"` is about 120 days, `"1d"` about 2 years. For more history, download Kraken's OHLCVT CSV files and use
`load_bars(symbol, "1d", csv_path="path/to/XBTEUR_1440.csv")`.

In [ ]:
SYMBOLS = ["BTC/EUR", "ETH/EUR", "SOL/EUR"]
INTERVAL = "1d"

data = {symbol: load_bars(symbol, INTERVAL) for symbol in SYMBOLS}
for symbol, bars in data.items():
    print(f"{symbol}: {len(bars)} bars, {bars[0].timestamp:%Y-%m-%d} -> {bars[-1].timestamp:%Y-%m-%d}")

## 2. Which strategies exist

In [ ]:
catalog_table()

## 3. Backtest one strategy

`run_strategy` runs one continuous backtest with costs (default: Kraken taker 0.40% + 10 bps slippage per fill,
about 1% per round trip), full position size and no risk overlay, then measures two periods:
**in-sample** (the first 70%) and **holdout** (the last 30%). `buy_hold_*` is the same period for simply holding the coin.

Things to try: another strategy name, other `params`, `"long_only": True` inside `params` (what spot live trading
effectively does), `costs=CostSettings(fee_pct=0.25, slippage_bps=0)` for limit orders.

In [ ]:
run = run_strategy(data["BTC/EUR"], "keltner_breakout", params={"window": 40, "atr_multiplier": 1.5})
run.metrics_table()

In [ ]:
plot_run(run);

## 4. Write your own strategy

A strategy is a function `(history, index, current_bar) -> 1 | 0 | -1` that returns the **position to hold**, not an
order: keep returning 1 to stay long. The engine buys when the value turns 1 and sells when it goes back to 0.

The easy way to write "enter when X, hold until Y":
1. compute indicator arrays over `history` with `src.backtest.indicators` (they are NaN until there is enough history,
   and NaN comparisons are False, so rules are automatically off during warmup),
2. build boolean arrays for your entry and exit rules,
3. let `latch_position(long_entry, long_exit, short_entry, short_exit)` work out whether you are in a trade now.

Wrap it in a factory (a function that takes the parameters) so it can be swept. The example: a Donchian breakout that
only goes long while price is above its long-term average.

In [ ]:
def breakout_with_trend_filter(entry_window=20, exit_window=10, trend_window=100):
    def strategy(history, index, current_bar):
        if len(history) < max(entry_window, trend_window) + 1:
            return 0
        close, high, low = series(history, "close"), series(history, "high"), series(history, "low")
        uptrend = close > rolling_mean(close, trend_window)
        return latch_position(
            long_entry=uptrend & (close > shift(rolling_max(high, entry_window))),
            long_exit=(close < shift(rolling_min(low, exit_window))) | ~uptrend,
        )

    return strategy


custom_run = run_strategy(data["BTC/EUR"], breakout_with_trend_filter(), measure_start=102, label="breakout+trend")
custom_run.metrics_table()

In [ ]:
custom = sweep(data, breakout_with_trend_filter, grid={"entry_window": [10, 20, 40], "trend_window": [50, 100]}, long_only=True)
summarize(custom)

If a custom strategy earns its place, move the factory into `src/backtest/strategies.py`, register it in
`StrategyRegistry` (`src/backtest/runner.py`) and add a `StrategySpec` in `src/research/catalog.py`. See the guide.

## 5. Sensitivity sweep

`sweep` runs every combination in the strategy's catalog grid on every symbol, in both long-only and long/short mode.
`summarize` condenses it to one verdict per strategy and side. The heatmap averages the symbols.

What robust looks like: a **broad block of blue in-sample that is still blue in the holdout panel**, a
`best_neighbors_is` close to `best_is`, a high `share_positive_is`, and holdout numbers that hold up against
`buy_hold_ho`. A single bright cell surrounded by red is luck.

In [ ]:
results = sweep(data, "keltner_breakout")
summarize(results)

In [ ]:
plot_heatmap(results, "keltner_breakout", long_only=False);

## 6. Compare every strategy at its defaults

A quick first pass. All strategies are measured from the same bar (102, the longest warmup), so their numbers line
up with each other and with buy-and-hold.

In [ ]:
overview = compare(data, measure_start=102)
overview.groupby(["strategy", "long_only"])[["is_sharpe", "ho_sharpe", "is_trades", "is_max_drawdown", "is_buy_hold_sharpe", "is_buy_hold_max_drawdown"]].mean().sort_values("is_sharpe", ascending=False)

## 7. How much do costs matter?

At ~1% per round trip a strategy needs a large average gain per trade. Compare the same sweep at taker fees,
limit-order (maker) fees and zero cost: if a strategy only works at zero cost, the signal is real but not tradable
at our fee tier.

In [ ]:
cost_levels = {"taker ~1.0%": CostSettings(), "maker ~0.5%": CostSettings(fee_pct=0.25, slippage_bps=0.0), "zero": CostSettings(fee_pct=0.0, slippage_bps=0.0)}
rows = []
for label, costs in cost_levels.items():
    summary = summarize(sweep(data, "moving_average_crossover", long_only=True, costs=costs))
    rows.append({"costs": label, "median_is": summary["median_is"].iloc[0], "median_ho": summary["median_ho"].iloc[0], "share_positive_is": summary["share_positive_is"].iloc[0]})
pd.DataFrame(rows)

## 8. Load results saved by the CLI

`python scripts/research/research.py sweep ...` writes `results.csv` and `summary.csv` to `data/research/<run>/`.
Load them here to slice them any way you like.

In [ ]:
saved_runs = sorted(Path("data/research").glob("sweep_*/results.csv"), key=lambda path: path.stat().st_mtime)
if saved_runs:
    latest = saved_runs[-1]  # or pick one explicitly, e.g. Path("data/research/sweep_20260925_full/results.csv")
    saved = pd.read_csv(latest)
    print(f"Loaded {latest} ({len(saved)} rows)")
    display(summarize(saved[saved["interval"] == "1d"]).head(10))
else:
    print("No saved sweeps yet - run: python scripts/research/research.py sweep")